# Module 2 · Lesson 07: Resume Customizer

Apply prompt engineering to a real career task: **tailoring resume bullet points**
to match a specific job description. This exercise ties together zero-shot, few-shot,
and constraint-based prompting.

## What you will learn
1. **System prompt design** for domain-specific tasks
2. **Few-shot examples** to control output style
3. **Constraint prompting** for length and format control
4. Iterative prompt refinement workflow

In [1]:
import os, json
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display, Markdown
 
load_dotenv(Path.cwd().parent / ".env")
 
from openai import OpenAI
client = OpenAI()
 
def ask(prompt, system=None, temperature=0.7, max_tokens=500):
    msgs = []
    if system:
        msgs.append({"role": "system", "content": system})
    msgs.append({"role": "user", "content": prompt})
    r = client.chat.completions.create(
        model="gpt-4o-mini", messages=msgs,
        temperature=temperature, max_tokens=max_tokens
    )
    return r.choices[0].message.content
 
print("Ready")

Ready


---
## 1. Define the Input

Two inputs: the **job description** (target) and your **work experience** (source).

In [2]:
job_description = """Senior Python Developer - AI Team
 
We're looking for an experienced Python developer to join our AI team.
You' ll build production ML pipelines, optimize model serving infrastructure,
and collaborate with data scientists.
 
Requirements:
- 5+ years of Python experience
- Experience with FastAPI or Flask
- ML/AI deployment (Docker, Kubernetes)
- Strong testing practices (pytestm CI/CD)
- Experience with cloud platforms (AWS/GCP)
"""
 
work_experience = """- Worked on a web application using Django for 3 years
- Helped deploy models to production servers
- Wrote some unit tests for the backed
- Used AWS for hosting and S3 for storage
- Collaborated with the data team on data pipelines
- Built REST APIs for internal tools"""

---
## 2. Zero-Shot Approach

First attempt: just ask the model to rewrite, no examples.

In [5]:
ZERO_SHOT_PROMPT = """You are senior hiring manager and resume expert.
Rewrite the candidate's experience bullets to better match the target job description.
 
Rules:
- Keep the same facts, but reframe using job's language
- Start each bullet with a strong action verb
- Include qualifiable results where possible
- Maximum 1 line per bullet point
- Do NOT fabricate experience
"""
 
zero_shot = ask(
    f"""Job description:
{job_description}
   
Original Experience:
{work_experience}
 
Rewritten bullets:""",
system = ZERO_SHOT_PROMPT,
temperature = 0.5
)

display(Markdown(f"### Zero-shot result:\n\n{zero_shot}"))

### Zero-shot result:

- Developed and maintained web applications using Django, leveraging 3 years of Python expertise.  
- Deployed machine learning models to production servers, ensuring robust model serving infrastructure.  
- Implemented comprehensive unit tests for backend components, enhancing code quality and reliability.  
- Utilized AWS for cloud hosting and S3 for efficient data storage solutions.  
- Collaborated closely with data scientists to optimize data pipelines for seamless integration.  
- Designed and built RESTful APIs for internal tools, streamlining development processes.  

---
## 3. Few-Shot Approach

Now let's provide **examples** of good rewrites to guide the style:

In [6]:
FEW_SHOT_SYSTEM = """You are a senior hiring manager and resume expert.
Rewrite experience bullets to match a target job description.
 
Here are examples of good rewrites:
 
BEFORE: "Helped with database stuff"
AFTER: "Optimized PostgreSQL query performance, reducing p95 latency by 40% acrross 3 production services"
 
BEFORE: "Made some APIs"
AFTER: "Architected and deployed RESTful APIs serving 50K+ daily requests using FastAPI with async patterns"
 
BEFORE: "Did testing"
AFTER: "Established comprehensive test suite with 92% coverage using pytest, integrated into CI/CD pipeline"
 
Rules:
- Match the STYLE of the examples above
- Use the job description's specific keywords
- Start each bullet with a STRONG action verb
- Add plausible metrics (but mark estimated ones with ~)
- Do NOT fabricate experience - only reframe existing facts"""

In [7]:
few_shot = ask(
    f"""Job description:
{job_description}
   
Original Experience:
{work_experience}
 
Rewritten bullets:""",
system = FEW_SHOT_SYSTEM,
temperature = 0.5
)
 
display(Markdown(f"### Few-shot resutlt:\n\n{few_shot}"))

### Few-shot resutlt:

- Developed and maintained a high-performance web application using Django, enhancing user engagement by ~30% over 3 years.

- Deployed machine learning models to production environments using Docker and Kubernetes, improving model availability and scalability by ~50%.

- Implemented a robust suite of unit tests achieving 85% coverage with pytest, integrated into CI/CD pipelines to ensure continuous delivery of high-quality code.

- Leveraged AWS for hosting and utilized S3 for efficient data storage, optimizing costs by ~20% while ensuring high availability.

- Collaborated with data scientists to streamline data pipelines, reducing data processing time by ~40% and improving model training efficiency.

- Designed and developed RESTful APIs for internal tools using FastAPI, facilitating seamless integration and serving ~10K daily requests.

---
## 4. Compare Results

In [8]:
compare = f""" ### Comparison
 
#### Original
{work_experience}
 
#### Zero-Shot Rewrite
{zero_shot}
 
#### Few-Shot Rewrite
{few_shot}
"""
 
display(Markdown(compare))

 ### Comparison

#### Original
- Worked on a web application using Django for 3 years
- Helped deploy models to production servers
- Wrote some unit tests for the backed
- Used AWS for hosting and S3 for storage
- Collaborated with the data team on data pipelines
- Built REST APIs for internal tools

#### Zero-Shot Rewrite
- Developed and maintained web applications using Django, leveraging 3 years of Python expertise.  
- Deployed machine learning models to production servers, ensuring robust model serving infrastructure.  
- Implemented comprehensive unit tests for backend components, enhancing code quality and reliability.  
- Utilized AWS for cloud hosting and S3 for efficient data storage solutions.  
- Collaborated closely with data scientists to optimize data pipelines for seamless integration.  
- Designed and built RESTful APIs for internal tools, streamlining development processes.  

#### Few-Shot Rewrite
- Developed and maintained a high-performance web application using Django, enhancing user engagement by ~30% over 3 years.

- Deployed machine learning models to production environments using Docker and Kubernetes, improving model availability and scalability by ~50%.

- Implemented a robust suite of unit tests achieving 85% coverage with pytest, integrated into CI/CD pipelines to ensure continuous delivery of high-quality code.

- Leveraged AWS for hosting and utilized S3 for efficient data storage, optimizing costs by ~20% while ensuring high availability.

- Collaborated with data scientists to streamline data pipelines, reducing data processing time by ~40% and improving model training efficiency.

- Designed and developed RESTful APIs for internal tools using FastAPI, facilitating seamless integration and serving ~10K daily requests.


In [9]:
coverage = ask(
    f"""Analyse how well this resume matched the job description
 
Job Description:
{job_description}
 
Resume Bullets:
{few_shot}
 
Return JSON with:
- "matched_keywords": list of JD keywords found in resume
- "missing_keywords": list of JD keywords NOT addressed
- "match_score": percentage (0-100)
- "suggestions": list of 2-3 improvements
""",
temperature = 0
)
 
display(Markdown(coverage))

```json
{
  "matched_keywords": [
    "5+ years of Python experience",
    "FastAPI",
    "ML/AI deployment",
    "Docker",
    "Kubernetes",
    "strong testing practices",
    "pytest",
    "CI/CD",
    "cloud platforms",
    "AWS"
  ],
  "missing_keywords": [
    "Flask",
    "GCP"
  ],
  "match_score": 90,
  "suggestions": [
    "Include experience with Flask to cover all web frameworks mentioned in the job description.",
    "Mention any experience with GCP to demonstrate familiarity with multiple cloud platforms.",
    "Highlight specific projects or achievements related to building production ML pipelines to align more closely with the job focus."
  ]
}
```

---
## 5. Structured Output: Keyword Matching

Let's also check which job requirements are covered:

---
## Key Takeaways

| Technique | Effect |
|-----------|--------|
| **Zero-shot** | Quick but generic — good starting point |
| **Few-shot examples** | Controls output style and quality significantly |
| **Negative constraints** | "Do NOT fabricate" prevents hallucinated experience |
| **Keyword matching** | Structured JSON output to verify coverage |

> **Exercise:** Replace the sample job description with a real one you're interested in.
> Add your own work experience. Experiment with different few-shot examples to find
> the voice that works best for your industry.

---
**Next:** `module_03_ai_architecture` — Build production-ready AI applications